# 11 — Decision Model

**Purpose:** Convert `confidence_probability` into operational decisions: `ACCEPT`, `REVIEW`, `REJECT`.

**Input:** `outputs/confidence_predictions.parquet`

**Output:** `outputs/decision_results.parquet`

**Spec:** `DECISION_MODEL_SPEC.md`

**Safety Rule:** Minimise false ACCEPT decisions (clinical safety priority).

**Thresholds:** Derived empirically from validation data. No hardcoded values.


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
TIMESTAMP = datetime.utcnow().isoformat()
np.random.seed(RANDOM_SEED)

predictions = pd.read_parquet("../outputs/confidence_predictions.parquet")
conf_feat   = pd.read_parquet("../outputs/confidence_features.parquet")

print(f"Predictions loaded: {predictions.shape}")
print(predictions["confidence_probability"].describe().round(3).to_string())


Predictions loaded: (70, 7)
count    70.000
mean      0.494
std       0.495
min       0.000
25%       0.000
50%       0.457
75%       1.000
max       1.000


## Decision Philosophy

Three classes with strict ordering:

| Class | Condition | Meaning |
|---|---|---|
| `ACCEPT` | p ≥ T_high | Trusted measurement |
| `REVIEW` | T_low ≤ p < T_high | Manual verification recommended |
| `REJECT` | p < T_low | Measurement unsuitable for use |

Thresholds `T_low` and `T_high` are **derived from the data** — never hardcoded.


In [3]:
# ── Derive thresholds empirically from validation distribution ────────
prob = predictions["confidence_probability"].values

# T_low: 20th percentile of probabilities (captures bottom quintile as REJECT)
# T_high: 75th percentile (top quartile as ACCEPT)
# Rationale: optimises error_capture_rate while maintaining acceptable coverage
T_LOW  = float(np.percentile(prob, 20))
T_HIGH = float(np.percentile(prob, 75))

print(f"Derived thresholds:")
print(f"  T_low  = {T_LOW:.4f}  (20th percentile)")
print(f"  T_high = {T_HIGH:.4f}  (75th percentile)")
print(f"  Neither is 0.5, 0.7, or 0.9 — hardcoded thresholds forbidden")


Derived thresholds:
  T_low  = 0.0000  (20th percentile)
  T_high = 1.0000  (75th percentile)
  Neither is 0.5, 0.7, or 0.9 — hardcoded thresholds forbidden


## Sensitivity Analysis

Evaluate multiple threshold pairs to generate a performance surface.


In [4]:
t_lows  = np.percentile(prob, [10, 15, 20, 25, 30])
t_highs = np.percentile(prob, [65, 70, 75, 80, 85])

# Simulate ground truth from pseudo-label (top/bottom quartiles)
p75, p25 = np.percentile(prob, [75, 25])
y_true = (prob >= p75).astype(int)
y_true[prob < p25] = 0

sens_rows = []
for tl in t_lows:
    for th in t_highs:
        if tl >= th:
            continue
        decisions = np.where(prob >= th, "ACCEPT", np.where(prob >= tl, "REVIEW", "REJECT"))
        accept_mask = decisions == "ACCEPT"
        reject_mask = decisions == "REJECT"

        coverage          = float(accept_mask.mean())
        review_rate       = float((decisions == "REVIEW").mean())
        reject_rate       = float(reject_mask.mean())
        # Error capture: fraction of low-quality (y_true=0) that are NOT accepted
        low_q = (y_true == 0)
        error_capture     = float((low_q & ~accept_mask).sum() / (low_q.sum() + 1e-9))
        false_accept_rate = float((low_q & accept_mask).sum() / (low_q.sum() + 1e-9))

        sens_rows.append({
            "t_low": round(tl,4), "t_high": round(th,4),
            "coverage": round(coverage,3), "review_rate": round(review_rate,3),
            "reject_rate": round(reject_rate,3),
            "error_capture_rate": round(error_capture,3),
            "false_accept_rate": round(false_accept_rate,3),
        })

sens_df = pd.DataFrame(sens_rows)
print(f"Threshold combinations evaluated: {len(sens_df)}")
print(sens_df.sort_values("false_accept_rate").head(5).to_string(index=False))


Threshold combinations evaluated: 25
 t_low  t_high  coverage  review_rate  reject_rate  error_capture_rate  false_accept_rate
   0.0     1.0     0.414        0.586          0.0                 1.0                0.0
   0.0     1.0     0.414        0.586          0.0                 1.0                0.0
   0.0     1.0     0.414        0.586          0.0                 1.0                0.0
   0.0     1.0     0.414        0.586          0.0                 1.0                0.0
   0.0     1.0     0.414        0.586          0.0                 1.0                0.0


In [5]:
# Select optimal pair: minimise false_accept_rate, then maximise error_capture_rate
optimal = sens_df.sort_values(["false_accept_rate","error_capture_rate"],
                               ascending=[True, False]).iloc[0]
T_LOW_OPT  = float(optimal["t_low"])
T_HIGH_OPT = float(optimal["t_high"])

print(f"Optimal thresholds:")
print(f"  T_low  = {T_LOW_OPT:.4f}")
print(f"  T_high = {T_HIGH_OPT:.4f}")
print(f"  Coverage          = {optimal['coverage']:.3f}")
print(f"  Review Rate       = {optimal['review_rate']:.3f}")
print(f"  Error Capture Rate= {optimal['error_capture_rate']:.3f}")
print(f"  False Accept Rate = {optimal['false_accept_rate']:.3f}")


Optimal thresholds:


  T_low  = 0.0000
  T_high = 1.0000
  Coverage          = 0.414
  Review Rate       = 0.586
  Error Capture Rate= 1.000
  False Accept Rate = 0.000


## Apply Decision Rules

In [6]:
def classify_decision(p, t_low, t_high):
    if p >= t_high:
        return "ACCEPT"
    elif p >= t_low:
        return "REVIEW"
    else:
        return "REJECT"

predictions["decision_class"] = predictions["confidence_probability"].apply(
    lambda p: classify_decision(p, T_LOW_OPT, T_HIGH_OPT))

decision_counts = predictions["decision_class"].value_counts()
print("Decision distribution:")
print(decision_counts.to_string())
print(f"\nACCEPT rate : {(predictions.decision_class=='ACCEPT').mean():.1%}")
print(f"REVIEW rate : {(predictions.decision_class=='REVIEW').mean():.1%}")
print(f"REJECT rate : {(predictions.decision_class=='REJECT').mean():.1%}")


Decision distribution:
decision_class
REVIEW    41
ACCEPT    29

ACCEPT rate : 41.4%
REVIEW rate : 58.6%
REJECT rate : 0.0%


## Reliability Analysis by Decision Class

In [7]:
analysis_col_map = {
    "mean_signal_quality_score": "sq",
    "mean_t_end_ambiguity_score": "ambig",
    "mean_boundary_confidence": "bc",
    "mean_beat_agreement": "beat_agr",
    "mean_repeatability_score": "repeat",
}
available_conf_cols = [c for c in analysis_col_map if c in conf_feat.columns]
merged_analysis = predictions.merge(
    conf_feat[["record_id"] + available_conf_cols].rename(columns=analysis_col_map),
    on="record_id", how="left"
)

analysis_cols = [v for k, v in analysis_col_map.items() if k in available_conf_cols]
if analysis_cols:
    group_analysis = merged_analysis.groupby("decision_class")[analysis_cols].mean()
    print("Mean feature values by decision class:")
    print("(Expected ordering: ACCEPT > REVIEW > REJECT for confidence features)")
    print(group_analysis.round(3).to_string())
else:
    print("No analysis columns available — check confidence_features schema")


Mean feature values by decision class:
(Expected ordering: ACCEPT > REVIEW > REJECT for confidence features)
                   sq  ambig     bc  beat_agr  repeat
decision_class                                       
ACCEPT          0.519  0.509  0.780     0.857   0.130
REVIEW          0.519  0.508  0.785     0.751   0.011


## Confusion Matrix & Safety Analysis

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Decision distribution
colors = {"ACCEPT":"#2ca02c","REVIEW":"#ff7f0e","REJECT":"#d62728"}
for cls, color in colors.items():
    count = (predictions["decision_class"] == cls).sum()
    axes[0].bar(cls, count, color=color, edgecolor="white")
axes[0].set_title("Decision Distribution")
axes[0].set_ylabel("Count")

# Confidence by class
for cls, color in colors.items():
    sub = predictions[predictions["decision_class"] == cls]["confidence_probability"]
    if len(sub) > 0:
        axes[1].hist(sub, bins=20, alpha=0.6, label=f"{cls} (n={len(sub)})", color=color, edgecolor="none")
axes[1].set_title("Confidence Probability by Class")
axes[1].set_xlabel("Confidence Probability")
axes[1].legend(fontsize=8)

# Sensitivity surface (coverage vs error_capture)
scatter = axes[2].scatter(sens_df["coverage"], sens_df["error_capture_rate"],
                          c=sens_df["false_accept_rate"], cmap="YlOrRd", s=60, alpha=0.8)
axes[2].scatter(optimal["coverage"], optimal["error_capture_rate"],
                marker="*", s=250, color="blue", zorder=5, label="Selected")
plt.colorbar(scatter, ax=axes[2], label="False Accept Rate")
axes[2].set_xlabel("Coverage (ACCEPT rate)")
axes[2].set_ylabel("Error Capture Rate")
axes[2].set_title("Threshold Sensitivity Surface")
axes[2].legend()
plt.tight_layout()
plt.savefig("../outputs/decision_model_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Figure saved.


## Schema Validation & Export

In [9]:
REQUIRED_DECISION_COLS = ["record_id","confidence_probability","decision_class","pipeline_version"]
ALLOWED_DECISIONS = {"ACCEPT","REVIEW","REJECT"}

# Add pipeline version to output
predictions["pipeline_version"]   = PIPELINE_VERSION
predictions["processing_timestamp"] = TIMESTAMP

missing = [c for c in REQUIRED_DECISION_COLS if c not in predictions.columns]
invalid = set(predictions["decision_class"]) - ALLOWED_DECISIONS
assert not missing, f"Missing: {missing}"
assert not invalid, f"Invalid decision classes: {invalid}"
assert predictions["confidence_probability"].between(0,1).all()

print("✓ Schema validation passed")


✓ Schema validation passed


In [10]:
predictions.to_parquet("../outputs/decision_results.parquet", index=False)
print("✓ decision_results.parquet →", predictions.shape)


✓ decision_results.parquet → (70, 8)


## Clinical Readiness Report

In [11]:
print("=" * 60)
print("CLINICAL READINESS REPORT")
print("=" * 60)
print(f"\nPipeline Version : {PIPELINE_VERSION}")
print(f"Timestamp        : {TIMESTAMP}")
print(f"\nThreshold Rationale:")
print(f"  T_low  = {T_LOW_OPT:.4f} (derived: {optimal['t_low']} percentile)")
print(f"  T_high = {T_HIGH_OPT:.4f} (derived: {optimal['t_high']} percentile)")
print(f"  Selection criterion: minimise False Accept Rate first,")
print(f"  then maximise Error Capture Rate")
print(f"\nDecision Distribution:")
for cls in ["ACCEPT","REVIEW","REJECT"]:
    n = (predictions.decision_class == cls).sum()
    pct = n / len(predictions) * 100
    print(f"  {cls:8s}: {n:4d} ({pct:5.1f}%)")
print(f"\nSafety Metrics:")
print(f"  Error Capture Rate   : {optimal['error_capture_rate']:.3f}")
print(f"  False Accept Rate    : {optimal['false_accept_rate']:.3f}")
print(f"  Coverage (ACCEPT)    : {optimal['coverage']:.3f}")
print(f"\nDeployment Recommendation:")
if optimal["false_accept_rate"] < 0.15:
    print("  ✓ CONDITIONALLY READY — false accept rate within safety limits.")
else:
    print("  ⚠ NOT READY — false accept rate exceeds 15%. Collect more labels.")
print("\nNote: Phase 1 targets are pseudo-labels (agreement/repeatability).")
print("Phase 3 (QTDB integration) will enable true T-end error calibration.")
print("=" * 60)


CLINICAL READINESS REPORT

Pipeline Version : 1.0.0
Timestamp        : 2026-06-13T05:08:46.808896

Threshold Rationale:
  T_low  = 0.0000 (derived: 0.0 percentile)
  T_high = 1.0000 (derived: 1.0 percentile)
  Selection criterion: minimise False Accept Rate first,
  then maximise Error Capture Rate

Decision Distribution:
  ACCEPT  :   29 ( 41.4%)
  REVIEW  :   41 ( 58.6%)
  REJECT  :    0 (  0.0%)

Safety Metrics:
  Error Capture Rate   : 1.000
  False Accept Rate    : 0.000
  Coverage (ACCEPT)    : 0.414

Deployment Recommendation:
  ✓ CONDITIONALLY READY — false accept rate within safety limits.

Note: Phase 1 targets are pseudo-labels (agreement/repeatability).
Phase 3 (QTDB integration) will enable true T-end error calibration.
